# ⚡ OmniFast Browser - Cloud Decision Server

**OmniFast Browser** is an open-source, ultra-fast hybrid browser agent that combines:
- **Decision Engines**: Proprietary TypeSafe Jev ↔ Free Google Colab Laya ↔ Local GPU/CPU
- **Browser Engines**: Playwright Chromium ↔ Lightpanda (Zig-based 10x faster headless browser)

This notebook runs the **Laya (ModernBERT-large)** non-autoregressive decision model on **Google Colab's Free GPU/16GB RAM** and exposes an instant, zero-configuration Cloudflare HTTPS tunnel.

---

In [ ]:
# @title ▶️ Step 1: Start OmniFast Decision Server (Click Run)
# 1. Install dependencies
!pip install -q laya

# 2. Install Cloudflare Tunnel (Free, zero signup, zero token)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import subprocess, time, json, re, threading
from http.server import HTTPServer, BaseHTTPRequestHandler
from laya import Router

print("[OmniFast] Initializing Laya Decision Engine...")
router = Router()

# Pre-warm model cache for instant inference
print("[OmniFast] Pre-warming model cache...")
router.predict("Warmup URL: https://example.com", {"op": {"type": "choice", "criteria": {"DONE": "Done"}}})
print("[OmniFast] Engine ready for sub-30ms decisions!")

class OmniFastHandler(BaseHTTPRequestHandler):
    def _send_json(self, status, data):
        body = json.dumps(data).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Headers", "*")
        self.send_header("Access-Control-Allow-Methods", "POST, GET, OPTIONS")
        self.end_headers()
        self.wfile.write(body)

    def do_OPTIONS(self):
        self.send_response(204)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Headers", "*")
        self.send_header("Access-Control-Allow-Methods", "POST, GET, OPTIONS")
        self.end_headers()

    def do_GET(self):
        self._send_json(200, {"status": "ok", "service": "omnifast-decision-server", "engine": "laya-multilingual"})

    def do_POST(self):
        content_length = int(self.headers.get("Content-Length", 0))
        body = json.loads(self.rfile.read(content_length).decode("utf-8"))
        context = body.get("context", "")
        questions = body.get("questions", {})
        t0 = time.perf_counter()
        result = router.predict(context, questions)
        result["latency_ms"] = round((time.perf_counter() - t0) * 1000, 2)
        self._send_json(200, result)

def run_server():
    server = HTTPServer(("127.0.0.1", 8000), OmniFastHandler)
    server.serve_forever()

threading.Thread(target=run_server, daemon=True).start()

# Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True)

public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print("\n" + "=" * 65)
    print("🚀 OMNIFAST ENDPOINT READY:")
    print(public_url)
    print("=" * 65)
    print(f"\nSet this in your local machine terminal:\n")
    print(f"export LAYA_ENDPOINT={public_url}")
else:
    print("Could not start tunnel. Check logs.")